# 🏆 Maximum Quality GPT Language Model (~10.8M Parameters)

### Step 1: Confirm T4 GPU
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > **Save**.

In [ ]:
import os
import torch

# Check dataset availability
if not os.path.exists('combined_train.txt') and not os.path.exists('input.txt'):
    print("Neither combined_train.txt nor input.txt found. Downloading fallback Shakespeare...")
    !curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

print("GPU Active:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU found! Please enable GPU in Runtime.")

### Step 2: High-Capacity Architecture & Training Pipeline

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --------------------------------------------------------------------------
# HIGH-CAPACITY GPU HYPERPARAMETERS (NanoGPT Flagship Configuration)
# --------------------------------------------------------------------------
batch_size    = 64      # 64 parallel sequences per batch
block_size    = 256     # Context window: 256 characters (2x longer memory!)
max_iters     = 6000    # 6,000 optimization steps (~4-5 minutes on T4 GPU)
eval_interval = 500     # Evaluate every 500 steps
learning_rate = 3e-4    # Optimal learning rate for 10M parameter transformer
eval_iters    = 50
n_embd        = 384     # 384 embedding channels (rich semantic vectors)
n_head        = 6       # 6 multi-head attention heads (head_size = 64)
n_layer       = 6       # 6 transformer residual blocks
dropout       = 0.2     # Regularization against overfitting
# --------------------------------------------------------------------------

# 1. Load Dataset & Character Tokenizer
dataset_file = 'combined_train.txt' if os.path.exists('combined_train.txt') else 'input.txt'
print(f"Training with dataset: {dataset_file}")
with open(dataset_file, 'r', encoding='utf-8') as f:
    text = f.read().replace('\r\n', '\n').replace('\r', '')

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# 2. Neural Architecture
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = (q @ k.transpose(-2, -1)) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        return self.dropout(F.softmax(wei, dim=-1)) @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa   = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        return x + self.ffwd(self.ln2(x))

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.ln_f(self.blocks(self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=device))))
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_tokens, temperature=0.6, top_k=15):
        for _ in range(max_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

# 3. Instantiate Model
model = GPT().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Model Parameters: {total_params:,} (~10.8 Million)")
print(f"Context Window: {block_size} tokens | Embedding Dim: {n_embd} | Layers: {n_layer}")

### Step 3: Run 6,000 Step Training Loop (~4-5 mins)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Training 10.8M parameter GPT on {device.upper()} for {max_iters} steps...")
for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {step:4d}/{max_iters} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), 'model.pt')
print("Training Complete! Saved to model.pt")

### Step 4: Test Continuations & Download Weights

In [ ]:
prompts = [
    "User: What is your name?\nAssistant:",
    "User: What is Vishwas AI?\nAssistant:",
    "User: What is PyTorch?\nAssistant:"
]

for p in prompts:
    ctx = torch.tensor([encode(p)], dtype=torch.long, device=device)
    print(f"\n{'=' * 60}\nPROMPT: {p}\n{'=' * 60}")
    print(decode(model.generate(ctx, max_tokens=250, temperature=0.6)[0].tolist()))

# Download the model back to your PC
from google.colab import files
files.download('model.pt')